In [ ]:
conda create -n sti_cascat python=3.10 -y
conda activate sti_cascat

git clone https://github.com/deepomicslab/CASCAT.git
cd CASCAT

# Install the official dependencies for your platform.
# The following commands are from the repository's CUDA 11.8 setup.
pip install torch==2.1.1 torchvision==0.16.1 torchaudio==2.1.1 \
  --index-url https://download.pytorch.org/whl/cu118

pip install torch_geometric==2.6.1 pyg_lib torch_scatter \
  torch_sparse torch_cluster torch_spline_conv \
  -f https://data.pyg.org/whl/torch-2.1.0+cu118.html

pip install scanpy==1.10.1 matplotlib networkx scikit-misc \
  pydot pot numpy==1.26.4 scikit-learn==1.5.2

pip install numba==0.60.0 numba-scipy==0.4.0 \
  pandas==2.2.3 scipy==1.11.0 pyyaml==6.0.3

In [ ]:
from pathlib import Path
import scanpy as sc

base = Path.home() / "MERFISH" / "S1R1" / "STI_results"

adata = sc.read_h5ad(base / "S1R1_prepared.h5ad")

# Use original counts as CASCAT's expression input.
adata.X = adata.layers["counts"].copy()

# Keep spatial coordinates in the AnnData object.
assert "spatial" in adata.obsm
adata.write_h5ad(base / "S1R1_CASCAT_input.h5ad")

print(adata)
print("Saved:", base / "S1R1_CASCAT_input.h5ad")

In [ ]:

from pathlib import Path
import scanpy as sc
import matplotlib.pyplot as plt
import numpy as np
import scipy.sparse as sp

RESULT_FILE = Path("./result/data_processed.h5ad")
OUT = Path("./S1R1_CASCAT_plots")
OUT.mkdir(parents=True, exist_ok=True)

adata = sc.read_h5ad(RESULT_FILE)
print(adata)
print("obs columns:", adata.obs.columns.tolist())
print("obsm keys:", list(adata.obsm.keys()))
print("uns keys:", list(adata.uns.keys()))

# Plot predicted clusters in spatial coordinates.
if "spatial" in adata.obsm and "cascat_clusters" in adata.obs:
    xy = adata.obsm["spatial"]
    fig, ax = plt.subplots(figsize=(8, 7))
    scat = ax.scatter(
        xy[:, 0], xy[:, 1],
        c=adata.obs["cascat_clusters"].astype("category").cat.codes,
        s=3, cmap="tab20"
    )
    ax.invert_yaxis()
    ax.set_aspect("equal")
    ax.set_title("CASCAT predicted clusters")
    fig.colorbar(scat, ax=ax, label="Encoded cluster")
    fig.tight_layout()
    fig.savefig(OUT / "01_clusters.png", dpi=200)
    plt.show()

# Inspect trajectory connectivity.
conn = adata.uns.get("cascat_connectivities")
if conn is not None:
    if sp.issparse(conn):
        conn = conn.toarray()
    plt.figure(figsize=(8, 7))
    plt.imshow(conn, cmap="magma", aspect="auto")
    plt.colorbar(label="Connectivity weight")
    plt.title("CASCAT trajectory connectivity")
    plt.xlabel("Cluster or graph node")
    plt.ylabel("Cluster or graph node")
    plt.tight_layout()
    plt.savefig(OUT / "02_connectivity.png", dpi=200)
    plt.show()

# Save all cell-level annotations for downstream plotting.
adata.obs.to_csv(OUT / "cell_results.csv")
print("CASCAT plots saved to:", OUT)
